# Example: Optimizing model parameters

The following notebook provides instructions and code on how to optimize model parameters.

In [49]:
from garisom_tools import SperryModel
from garisom_tools.optimization import GarisomOptimizationConfig, Optimizer

from datetime import datetime
import ray
import os
import pandas as pd

## Setup model

In [50]:
# Absolute path directory links needed
model_dir = os.path.abspath("./garisom/02_program_code/")
param_directory = os.path.abspath("./DBG/")

In [51]:
# Set population (corresponds to line in parameters.csv file)
population = 1

In [52]:
# Function to get parameter and configuration files
def get_parameter_and_configuration_files(base_dir: str) -> tuple[str, pd.DataFrame]:
    return os.path.abspath(os.path.join(base_dir, "configuration.csv")), \
        pd.read_csv(os.path.join(base_dir, "parameters.csv"))

model_config, params = get_parameter_and_configuration_files(param_directory)

In [53]:
# Fetch the ground truth csv file for population
def get_ground_truth(population: int):
    ground_dir = "./files/ground"
    match population:
        case 1:
            ground = pd.read_csv(os.path.abspath(f"{ground_dir}/ccr_hourly_data.csv"))
        case 2:
            ground = pd.read_csv(os.path.abspath(f"{ground_dir}/jla_hourly_data.csv"))
        case 3:
            ground = pd.read_csv(os.path.abspath(f"{ground_dir}/nrv_hourly_data.csv"))
        case 4:
            ground = pd.read_csv(os.path.abspath(f"{ground_dir}/tsz_hourly_data.csv"))
        case _:
            raise Exception("Incorrect POP_NUM!")

    return ground
ground = get_ground_truth(population)

In [ ]:
# Necessary start and date datetime instances for evaluation and optimization
start_date = datetime(2023, 7, 20)
end_date = datetime(2023, 8, 24)

In [55]:
# When running with the Optimizer, it is necessary to pass in the following run_kwargs and eval_kwargs
run_kwargs = {
    'params': params,
    'config_file': model_config,
    'model_dir': model_dir,
    'population': population,
    'verbose': False
}
eval_kwargs = {
    'ground': ground,
    'start_date': start_date,
    'end_date': end_date
}

In [56]:
model = SperryModel(run_kwargs=run_kwargs, eval_kwargs=eval_kwargs)  # create model

## Setup optimization

In [57]:
# Create a metric config (this is how we define what outputs to evaluate and with what metrics)
metric_config = {
    'params': ['GW.a', 'GW.b', 'leaftemp'],  # 'params' can include suffixes (e.g., 'output_var.a', 'output_var.b') for multiple metrics on same output
    'metrics': ['rmse', 'mape', 'nnse'],  # Metric names must be supported by the Metric.from_name() method, see garisom_tools/utils/metric.py for a full list
    'modes': ['min', 'min', 'max']  # either 'min' or 'max' (either minimize or maximize error)
}

In [58]:
# Create a space config (this is how we define the search space for parameters)
# Three distributions
#   - uniform(low, high), Uniform distribution
#   - norm(mu, sigma), Normal distribution
#   - truncnorm(mu, sigma, low, high), Truncated normal distribution
space_config = {
    'i_fieldCapFrac': ['uniform', [0.5, 1]],
    'i_fieldCapPercInit': ['uniform', [50, 100]],
    'i_rootBeta': ['uniform', [0.8, 1]],
    'i_leafAreaIndex': ['truncnorm', [3.7, 0.5, 1, 5]]
}

In [ ]:
# Create optimization config, consists of space_config, metric_config, and assorted optimization parameters.
optim_config_dict = {
    'space': space_config,
    'metric': metric_config,
    'num_workers': -1,
    'num_samples': 40,   # number of samples to use, general rule is n = ~10 * # of parameters, may need to use more
    'population': population,
    'start_date': start_date,
    'end_date': end_date
}

In [60]:
optim_config = GarisomOptimizationConfig.from_dict(optim_config_dict)

In [61]:
# Ensure number of workers matches config, '-1' means choose max given CPU cores.
ray.shutdown()
if optim_config.num_workers == -1:
    ray.init()
else:
    ray.init(num_cpus=optim_config.num_workers)
print(ray.cluster_resources())  # show resources available

2026-05-09 13:55:50,750	INFO worker.py:2023 -- Started a local Ray instance.


{'node:127.0.0.1': 1.0, 'object_store_memory': 2147483648.0, 'CPU': 12.0, 'node:__internal_head__': 1.0, 'memory': 8016003072.0}


In [62]:
optim = Optimizer(
    model=model,
    config=optim_config,
    verbosity=1  # RayTune output verbosity level
)

[I 2026-05-09 13:55:51,520] A new study created in memory with name: optuna


## Run optimization

In [ ]:
param_results = optim.run()
# param_results.to_json(res_dir)  # saves parameter results as JSON file

(pid=gcs_server) [2026-05-09 13:56:20,120 E 18360 43078899] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2026-05-09 13:56:20,734 E 18364 43079006] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
[2026-05-09 13:56:21,529 E 17349 43079065] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(bundle_reservation_check_func pid=18372) [2026-05-09 13:56:21,464 E 18372 43079530] core_worker_process.cc:837: Failed to establish connection to the metrics expor

In [64]:
param_results.to_dict()

{'GW.a': {'scores': {'GW.a': 191.57713450948432,
   'GW.b': 0.6542276621202208,
   'leaftemp': 0.8188349413918585},
  'parameters': {'i_fieldCapFrac': 0.5490622945346904,
   'i_fieldCapPercInit': 61.799078444667394,
   'i_rootBeta': 0.9799379662392116,
   'i_leafAreaIndex': np.float64(3.093333764257592)}},
 'GW.b': {'scores': {'GW.a': 199.63091279915594,
   'GW.b': 0.6428193789598575,
   'leaftemp': 0.8039319917332639},
  'parameters': {'i_fieldCapFrac': 0.5571784008219189,
   'i_fieldCapPercInit': 62.38698734131988,
   'i_rootBeta': 0.9844535929414381,
   'i_leafAreaIndex': np.float64(3.100550014158294)}},
 'leaftemp': {'scores': {'GW.a': 194.59888103918405,
   'GW.b': 0.6725053438368404,
   'leaftemp': 0.8229110134117116},
  'parameters': {'i_fieldCapFrac': 0.5942572915365155,
   'i_fieldCapPercInit': 56.53622857565393,
   'i_rootBeta': 0.9775216178048591,
   'i_leafAreaIndex': np.float64(3.1164479105826373)}}}